# Transcription & Phonemic Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensein/senselab/blob/main/tutorials/audio/transcription_and_phonemic_analysis.ipynb)

In this tutorial you will learn to:
- Transcribe speech using automatic speech recognition (ASR / Whisper)
- Align transcriptions to audio at the word and phoneme level (forced alignment)
- Extract articulatory features using SPARC (speech articulatory coding)
- Extract phonetic posteriorgrams (PPGs)
- Analyze phoneme durations derived from PPGs
- Create time-aligned visualizations combining waveforms, spectrograms, and phonemic annotations

In [ ]:
# Install senselab and dependencies
!pip install -q uv
!uv pip install --pre --system "senselab[nlp]"

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages (e.g., numpy) that are already loaded.
> Go to **Runtime → Restart session**, then **run all cells below** (skip this install cell).

In [ ]:
import os
import urllib.request

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch

from senselab.audio.data_structures import Audio
from senselab.audio.tasks.features_extraction.ppg import (
    _extract_ppg_segments,
    _to_frame_major_posteriorgram,
    extract_mean_phoneme_durations,
    extract_ppgs_from_audios,
    plot_ppg_phoneme_timeline,
)
from senselab.audio.tasks.features_extraction.sparc import SparcFeatureExtractor
from senselab.audio.tasks.forced_alignment import align_transcriptions
from senselab.audio.tasks.plotting import play_audio, plot_specgram, plot_waveform
from senselab.audio.tasks.preprocessing.preprocessing import downmix_audios_to_mono, resample_audios
from senselab.audio.tasks.speech_to_text import transcribe_audios
from senselab.utils.data_structures import DeviceType, HFModel, Language, ScriptLine

# Auto-detect device
if torch.cuda.is_available():
    device = DeviceType.CUDA
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = DeviceType.CPU  # MPS not supported by all backends
else:
    device = DeviceType.CPU
print(f"Using device: {device.value}")

## 1. Record or Load Audio

We need a clear English speech sample. You can either **record your own voice**
in Google Colab, or use a **sample file** downloaded automatically.


In [ ]:
# Record audio in Google Colab (uses browser microphone)
# If not in Colab, skip this cell and run the next one to load a sample file.
from pathlib import Path

Path("tutorial_audio_files").mkdir(exist_ok=True)

try:
    from google.colab import output
    from IPython.display import HTML, display

    RECORD_JS = """
    <button id="record-btn" style="font-size:16px;padding:8px 16px">🎙 Click to Record (5s)</button>
    <script>
    (async () => {
        const btn = document.getElementById('record-btn');
        btn.onclick = async () => {
            btn.textContent = '🔴 Recording...';
            const stream = await navigator.mediaDevices.getUserMedia({audio: true});
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            recorder.ondataavailable = e => chunks.push(e.data);
            recorder.onstop = async () => {
                stream.getTracks().forEach(t => t.stop());
                const blob = new Blob(chunks, {type: 'audio/wav'});
                const reader = new FileReader();
                reader.onload = () => {
                    const b64 = reader.result.split(',')[1];
                    google.colab.kernel.invokeFunction('save_recording', [b64], {});
                };
                reader.readAsDataURL(blob);
                btn.textContent = '✅ Recorded! Run next cell to load.';
            };
            recorder.start();
            setTimeout(() => recorder.stop(), 5000);
        };
    })();
    </script>
    """

    def save_recording(b64_data):
        import base64

        raw = base64.b64decode(b64_data)
        with open("tutorial_audio_files/my_recording.wav", "wb") as f:
            f.write(raw)

    output.register_callback("save_recording", save_recording)
    display(HTML(RECORD_JS))
    print("Click the button to record 5 seconds of audio.")
    print("After recording, run the NEXT cell to load it.")
except Exception:
    print("Recording not available. Run the next cell to load a sample file.")

In [ ]:
# Load recorded audio if available, otherwise download a sample file

base_url = "https://github.com/sensein/senselab/raw/main/src/tests/data_for_testing"
dest = Path("tutorial_audio_files/audio_48khz_mono_16bits.wav")
if not dest.exists():
    urllib.request.urlretrieve(f"{base_url}/audio_48khz_mono_16bits.wav", str(dest))

# Check if a recording was made
rec_path = Path("tutorial_audio_files/my_recording.wav")
if rec_path.exists() and rec_path.stat().st_size > 1000:
    audio = Audio(filepath=str(rec_path.resolve()))
    print("Using your recording")
else:
    audio = Audio(filepath=os.path.abspath("tutorial_audio_files/audio_48khz_mono_16bits.wav"))
    print("Using sample audio")

audio_16k = resample_audios(downmix_audios_to_mono([audio]), resample_rate=16000)[0]
print(f"Duration: {audio_16k.waveform.shape[1] / 16000:.2f}s at 16kHz")
play_audio(audio_16k)

## 2. Automatic Speech Recognition

Automatic speech recognition (ASR) converts speech to text. We use OpenAI's Whisper,
a transformer model trained on 680,000 hours of multilingual audio. It takes audio as
input and produces a text transcription.

Senselab wraps the HuggingFace Transformers pipeline, so all you need to do is specify
the model and the audio.

In [ ]:
model = HFModel(path_or_uri="openai/whisper-small", revision="main")
transcripts = transcribe_audios(audios=[audio_16k], model=model, device=device)
transcript_text = transcripts[0].text
print(f"Transcription: {transcript_text}")

## 3. Forced Alignment

Given audio and its transcription, **forced alignment** finds the precise time boundaries
for each word and each phoneme (character). This uses a CTC-based wav2vec2 model that
learns the correspondence between audio frames and text characters.

The alignment result is a nested `ScriptLine` hierarchy:
- **Utterance** level: the entire transcription with start/end times
- **Sentence** level: sentences within the utterance
- **Word** level: individual words with timestamps
- **Character** level: individual characters (phonemes) with timestamps

In [ ]:
script = ScriptLine(text=transcript_text)
alignment_results = align_transcriptions(
    [(audio_16k, script, Language(language_code="en"))],
    levels_to_keep={"utterance": True, "word": True, "char": True},
)

# The result is a list (per audio) of lists (per segment) of ScriptLine objects.
# With utterance=True, we get the full utterance with nested sentences/words/chars.
utterance = alignment_results[0][0]

# Collect word-level items by walking the hierarchy:
# utterance -> sentences -> words -> chars
words = []
if utterance and utterance.chunks:
    for sentence in utterance.chunks:
        if sentence.chunks:
            for word in sentence.chunks:
                words.append(word)

print("=== Word-level Alignment ===")
for word in words:
    print(f"  [{word.start:.3f}s - {word.end:.3f}s] {word.text}")

## 4. Aligned Visualization

Visualizing word boundaries alongside the waveform and spectrogram gives a complete
picture of speech. You can see which words correspond to which acoustic patterns.

Below we create a three-panel aligned plot:
- **Top**: Waveform (amplitude over time)
- **Middle**: Mel spectrogram (frequency content over time)
- **Bottom**: Word boundaries as colored spans with character-level tick marks

In [ ]:
import torchaudio.transforms as T

waveform = audio_16k.waveform.squeeze().numpy()
sr = audio_16k.sampling_rate
duration = len(waveform) / sr
time_axis = np.linspace(0, duration, len(waveform))

# Compute mel spectrogram
spec_transform = T.MelSpectrogram(sample_rate=sr, n_fft=1024, hop_length=256, n_mels=80)
spec_db = T.AmplitudeToDB()(spec_transform(audio_16k.waveform.squeeze()))

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [1, 2, 1]})

# Panel 1: Waveform
axes[0].plot(time_axis, waveform, linewidth=0.3, color="steelblue")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Aligned Speech Analysis: Waveform + Spectrogram + Word Boundaries")
axes[0].grid(True, alpha=0.2)

# Panel 2: Mel Spectrogram
axes[1].imshow(
    spec_db.numpy(),
    aspect="auto",
    origin="lower",
    extent=[0, duration, 0, 8],
    cmap="magma",
)
axes[1].set_ylabel("Frequency (kHz)")

# Panel 3: Word and character boundaries
if words:
    colors = plt.get_cmap("Set3", max(len(words), 3))
    for i, word in enumerate(words):
        if word.start is not None and word.end is not None:
            axes[2].axvspan(word.start, word.end, alpha=0.3, color=colors(i % colors.N))
            axes[2].text(
                (word.start + word.end) / 2,
                0.5,
                word.text,
                ha="center",
                va="center",
                fontsize=7,
                fontweight="bold",
            )
            # Add character-level boundaries if available
            if word.chunks:
                for char in word.chunks:
                    if char.start is not None:
                        axes[2].axvline(char.start, color="gray", linewidth=0.5, alpha=0.5)

axes[2].set_ylabel("Words")
axes[2].set_xlabel("Time (seconds)")
axes[2].set_xlim(0, duration)
axes[2].set_yticks([])

plt.tight_layout()
plt.show()

## 5. Articulatory Coding (SPARC)

[SPARC](https://arxiv.org/abs/2406.12998) is a neural encoding-decoding
framework that represents speech as **14 articulatory features** at 50 Hz,
plus a disentangled speaker embedding. Each channel corresponds to a
physical articulator on the vocal tract:

- **Tongue Tip (TT)**, **Tongue Body (TB)**, **Tongue Dorsum (TD)**: x/y position
- **Lower Incisor (LI)** (jaw): x/y position
- **Upper Lip (UL)**, **Lower Lip (LL)**: x/y position

This gives us a *physical* representation of speech production, complementing
the *symbolic* (phonemic) representation from PPGs.

In [ ]:
# Extract SPARC articulatory features
sparc_features = SparcFeatureExtractor.extract_sparc_features(audios=[audio_16k], device=device, resample=True)
features = sparc_features[0]
print(f"EMA shape: {features['ema'].shape}  (channels x frames)")
print(f"Pitch shape: {features['pitch'].shape}")
print(f"Loudness shape: {features['loudness'].shape}")

### Articulatory Trajectories

The EMA features show estimated positions of 6 articulators (x/y each = 12 channels).
We plot them as time series, vertically offset for clarity.

In [ ]:
# Color scheme for articulators
COLOR_CODE = {
    "UL": matplotlib.colors.to_rgb("#EE3A5B"),
    "LL": matplotlib.colors.to_rgb("#FFD155"),
    "LI": matplotlib.colors.to_rgb("#959595"),
    "TT": matplotlib.colors.to_rgb("#43B96B"),
    "TB": matplotlib.colors.to_rgb("#3EADD8"),
    "TD": matplotlib.colors.to_rgb("#B075D0"),
}
CHANNEL_NAMES = [
    "TT_x",
    "TT_y",
    "TB_x",
    "TB_y",
    "TD_x",
    "TD_y",
    "UL_x",
    "UL_y",
    "LL_x",
    "LL_y",
    "LI_x",
    "LI_y",
]

ema = features["ema"].numpy()
n_channels = min(ema.shape[0], len(CHANNEL_NAMES))
n_frames = ema.shape[1]
time_ema = np.arange(n_frames) / 50.0  # SPARC runs at 50 Hz

fig, ax = plt.subplots(figsize=(14, 8))
offset = 0
for ch in range(n_channels):
    name = CHANNEL_NAMES[ch]
    artic = name.split("_")[0]
    color = COLOR_CODE.get(artic, (0.5, 0.5, 0.5))
    signal = ema[ch] - ema[ch].mean()
    ax.plot(time_ema, signal + offset, color=color, linewidth=0.8, label=name)
    offset += 3

ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Articulators (offset)")
ax.set_title("SPARC Articulatory Trajectories (EMA)")
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### SPARC Pitch and Loudness

SPARC also extracts pitch and loudness contours, providing a complementary
view of prosody alongside the articulatory features.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)

pitch = features["pitch"].numpy().squeeze()
loudness = features["loudness"].numpy().squeeze()
pitch_time = np.arange(len(pitch)) / 50.0
loud_time = np.arange(len(loudness)) / 50.0

axes[0].plot(pitch_time, pitch, linewidth=0.8, color="steelblue")
axes[0].set_ylabel("Pitch")
axes[0].set_title("SPARC Prosody Features")
axes[0].grid(True, alpha=0.3)

axes[1].plot(loud_time, loudness, linewidth=0.8, color="coral")
axes[1].set_ylabel("Loudness")
axes[1].set_xlabel("Time (seconds)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Phonetic Posteriorgrams (PPGs)

While forced alignment gives binary phoneme boundaries, **Phonetic Posteriorgrams (PPGs)**
provide a richer view: the probability of **each** phoneme at every time frame.

This captures the continuous, overlapping nature of speech sounds (coarticulation). PPGs
are extracted by a neural network that maps audio frames to phoneme probability distributions.

Note: PPG extraction uses an isolated subprocess virtual environment (because the `ppgs` library
has specific dependency requirements). The first run may take a few minutes to set up.

In [ ]:
ppgs = extract_ppgs_from_audios([audio_16k], device=device)
ppg = ppgs[0]
print(f"PPG shape: {ppg.shape}")
print("This gives us the probability of each phoneme at every time frame.")

## 7. Phoneme Duration Analysis

From the PPG, we can determine which phoneme is most likely at each frame (argmax),
group contiguous frames into segments, and calculate how long each phoneme lasts.

This reveals speaking rhythm, articulation speed, and phonemic patterns. The analysis
aggregates across all segments of the same phoneme to compute mean and total durations.

In [ ]:
durations = extract_mean_phoneme_durations(audio_16k, ppg)
print(f"Analysis: {durations['frame_count']} frames over {durations['analysis_duration_seconds']:.2f}s")
print("\n=== Top 10 Phonemes by Total Duration ===")
for p in sorted(durations["phoneme_durations"], key=lambda x: x["total_duration_seconds"], reverse=True)[:10]:
    bar = "\u2588" * int(p["total_duration_seconds"] * 50)
    print(f"  {p['phoneme']:>10s}: {p['mean_duration_seconds'] * 1000:6.1f}ms avg ({p['count']} segments) {bar}")

### Time-Aligned PPG Visualization

Below we align the PPG phoneme timeline with the waveform and spectrogram,
showing which phonemes correspond to which acoustic patterns in the signal.

In [ ]:
import torchaudio.transforms as T

waveform_np = audio_16k.waveform.squeeze().numpy()
sr = audio_16k.sampling_rate
total_duration = len(waveform_np) / sr
time_wav = np.linspace(0, total_duration, len(waveform_np))

# Get PPG segments for plotting
frame_major = _to_frame_major_posteriorgram(ppg)
segments = _extract_ppg_segments(audio_16k, frame_major)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [1, 2, 1.5]})

# Panel 1: Waveform
axes[0].plot(time_wav, waveform_np, linewidth=0.3, color="steelblue")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Aligned: Waveform + Spectrogram + PPG Phoneme Timeline")

# Panel 2: Spectrogram
spec_t = T.MelSpectrogram(sample_rate=sr, n_fft=1024, hop_length=256, n_mels=80)
spec_db = T.AmplitudeToDB()(spec_t(audio_16k.waveform.squeeze()))
axes[1].imshow(
    spec_db.numpy(), aspect="auto", origin="lower", extent=[0, total_duration, 0, sr / 2 / 1000], cmap="magma"
)
axes[1].set_ylabel("Frequency (kHz)")
axes[1].set_ylim(0, 8)

# Panel 3: PPG phoneme segments as colored bars
present_phonemes = sorted({s["phoneme"] for s in segments})
y_map = {p: i for i, p in enumerate(present_phonemes)}
colors = plt.get_cmap("tab20", len(present_phonemes))

for seg in segments:
    y = y_map[seg["phoneme"]]
    axes[2].barh(
        y, seg["duration_seconds"], left=seg["start_seconds"], height=0.7, color=colors(y), alpha=0.85, edgecolor="none"
    )

axes[2].set_yticks(range(len(present_phonemes)))
axes[2].set_yticklabels(present_phonemes, fontsize=7)
axes[2].set_ylabel("Phoneme")
axes[2].set_xlabel("Time (seconds)")
axes[2].set_xlim(0, total_duration)
axes[2].grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In this tutorial we covered the full pipeline from raw audio to detailed
phonemic and articulatory analysis:

| Analysis | Representation | senselab API |
|---|---|---|
| **ASR (Whisper)** | Text transcription | `transcribe_audios()` |
| **Forced Alignment** | Word/phone boundaries | `align_transcriptions()` |
| **SPARC** | Articulatory trajectories (physical space) | `SparcFeatureExtractor.extract_sparc_features()` |
| **PPGs** | Phoneme probabilities (symbolic space) | `extract_ppgs_from_audios()` |
| **Phoneme Durations** | Segment timing | `extract_mean_phoneme_durations()` |

These representations are complementary -- forced alignment gives precise
boundaries, SPARC shows the physical articulation, and PPGs capture the
continuous phonemic structure of speech.